# Mini Project 1 — Analysis Notebook

**Your name:**  Ruofu Li
**Dataset:**  Riot Developer API
**Date:**  May 20, 2026

In [4]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

# After you build a chart: fig = px.bar(...); fig.write_image("chart_name.png")
print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

**Dataset:** A combination of 3 different APIs from the Riot Developer API, including Summoner ID, Match by PUUID, and Match Details by Match ID. 

**Why this dataset:** I play League of Legends and I was curious about the champion picking habits of other players around my skill level. 

**Three analytical questions:**

1. Who is the most played Support champion in Bronze IV Solo 5x5 Ranked lobbies? 
2. What is the win rate for the top 3 Support champions in this lobby? 
3. Which Support champion has the highest win rate? 

**If time allows:** 

4. What is the distribution for the most played Support champion across all Level IV ranks (ex: Silver IV, Platinum IV, etc.)? 
5. What is the most common Bottom Lane/Support pairing in Bronze-level lobbies? What about across all ranks? 

*Some of these questions have been adjusted since the initial project proposal 

**What a practitioner would do with these findings:** Probably learn how to play other champions that best counter the most popular Support picks, or at least attempt to get better at them. 

---

## Section 2 — Data Profile

This section will focus on the analysis derived from a series of pandas operations to analyze the composition of the dataset being used. As a disclaimer, `support_picks.csv` has already been highly filtered as a result of the long process it took to even fetch that data. It's the culmination of 3 separate Riot Developer APIs (PUUIDS -> Match IDs -> Match Data), and had been further filtered to be more efficient for my device memory due to the sheer volume of data available. 

In [5]:
from pathlib import Path

data_path = Path("..") / "MP1" / "support_picks.csv"
df = pd.read_csv(data_path)

# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9631 entries, 0 to 9630
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   match_id            9631 non-null   str  
 1   puuid               9631 non-null   str  
 2   championId          9631 non-null   int64
 3   championName        9631 non-null   str  
 4   teamId              9631 non-null   int64
 5   teamPosition        9631 non-null   str  
 6   individualPosition  9631 non-null   str  
 7   win                 9631 non-null   bool 
dtypes: bool(1), int64(2), str(5)
memory usage: 536.2 KB


This operation tells us the kinds of information that is included in the dataset, in addition to how many values there are for each column and their data types. 

In [9]:
df.head()

,match_id,puuid,championId,championName,teamId,teamPosition,individualPosition,win
0,NA1_5550304472,UL3qXsy63hmThQw39LTtPWK2JI316v5jHX8r_yw5g9A873...,235,Senna,100,UTILITY,UTILITY,False
1,NA1_5550304472,o9UXKEi4u2kwH6F9UJVxFy8x6Jbi2tXl3Y5sFmPaDBR8Us...,89,Leona,200,UTILITY,UTILITY,True
2,NA1_5545162641,iAPisvb-m-cDkpVmzVJtNKdNjTwoIgk-PkKIvRQScvPeed...,53,Blitzcrank,100,UTILITY,UTILITY,True
3,NA1_5545162641,z47LTXaQxM72C5dDdLwpdhuZJVDjcmnqkjE11WKKhqtDPx...,26,Zilean,200,UTILITY,UTILITY,False
4,NA1_5530609519,Kj3raKvthe46haxKC6N2LOpUK8_VxgOyRUo48bufObv60i...,22,Ashe,100,UTILITY,UTILITY,True


I also wanted to run `df.tail()` so I could confirm if the correct DataFrame was being used.

In [ ]:
df.tail()

,match_id,puuid,championId,championName,teamId,teamPosition,individualPosition,win
9626,NA1_5540881381,y3qFob-7MxNeRF6bDyWUmt82ibYdfSDiZaNf1p68NPIUdu...,267,Nami,200,UTILITY,UTILITY,False
9627,NA1_5540069235,koymJUUO-Txq5txzIlr9UD-pRx0uAL54RpSs1fHYka-DVz...,117,Lulu,100,UTILITY,UTILITY,False
9628,NA1_5540069235,OQRjCk_JAveU_c_SBFVaBW-7mayej7kx3LKliGj_aKUN0H...,25,Morgana,200,UTILITY,UTILITY,True
9629,NA1_5539518433,YWgOXMk9hKyvCb5yVFNyXEA-FNCZxcP1017gbd9a5rJB-C...,235,Senna,100,UTILITY,UTILITY,True
9630,NA1_5539518433,idAFkFiSmr1P_-aUp4E7ksoUaocRqkyEBebdcuHEAwoC1j...,497,Rakan,200,UTILITY,UTILITY,False


Overall, `df.head()` and `df.tail()` both give us a sample of what the dataset looks like and visually presents them. It's consistent with the information that we saw in `df.info()`. 

In [7]:
# Summary statistics for numeric columns
df.describe()

#included from the template provided, but inapplicable to this dataset because it relies on string data instead of integers 

,championId,teamId
count,9631.000000,9631.000000
mean,191.906033,150.015575
std,215.779393,50.002594
min,1.000000,100.000000
25%,50.000000,100.000000
50%,99.000000,200.000000
75%,235.000000,200.000000
max,950.000000,200.000000


This operation was interesting because the numerical values included in my dataset have little analytical significance, at least in answering my questions. This is because the numbers being used are mostly identifiers. The data that actually matters for our analysis are all in strings and booleans, which are poorly represented with `df.describe()`.

In [11]:
df.isnull().sum()

match_id              0
puuid                 0
championId            0
championName          0
teamId                0
teamPosition          0
individualPosition    0
win                   0
dtype: int64

This operation shows us that there are no null values in this dataset. To reiterate, the source .csv file was highly filtered and cleaned prior to being used in this notebook. While working on it, I discovered that 4 Match IDs that were used to fetch Match Data contained empty dicts, and were consequently skipped when compiling the source .csv file for our analysis. You can refer to this in section 8ai in `MP1_dataset_prep.ipynb`.

#### Overall... 

- This dataset has 8 columns and 9631 rows, including a row for column titles. 
- Each column represents a different characteristic of each champion that was picked, including Match ID, PUUIDs for each player, champion ID associated with each champion, the champion's name, which team each player is on, team position, individual position, and if that player was part of the winning team. 
- No inconsistencies were identified, but I was a little confused on what the difference was between Individual Position and Team Position. They are all the same for each champion in the dataset (Utility for both)
- I'll be focusing on champion name, win, and probably team ID for my analysis. These fields will allow me to answer my questions concerning which Support champion is the most popular pick, win rates associated with these champions, and team composition the best. 

---

## Section 3 — Analysis

In this section, we'll finally be analyzing `support_picks.csv` to answer our core analytical questions.

#### Question 1: Who is the most played Support champion in Bronze IV Solo 5x5 Ranked lobbies? 
The script below groups all champions in `support_picks.csv` and aggregates their pick count to see which Support champion is the most selected, represented in descending order. 

In [12]:
# Count how often each support champion appears, then sort most → least
champion_pick_counts = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"))
    .sort_values("pick_count", ascending=False)
)

champion_pick_counts.head(10)

,championName,pick_count
67,Lux,805
77,Morgana,683
108,Seraphine,668
79,Nami,376
62,Leona,364
81,Nautilus,343
15,Brand,329
117,Sona,311
126,Thresh,302
92,Pyke,285


In [13]:
# Bar chart: top 10 Support pick counts (uses sorted data from the cell above)
if "champion_pick_counts" not in globals():
    raise RuntimeError("Run the cell above first so champion_pick_counts exists.")

champion_pick_chart = champion_pick_counts.nlargest(10, "pick_count").sort_values(
    "pick_count", ascending=False
)

fig = px.bar(
    champion_pick_chart,
    x="championName",
    y="pick_count",
    title="Lux is the most picked Support champion in Bronze IV ranked games",
    labels={"championName": "Support champion", "pick_count": "Pick count"},
    category_orders={"championName": champion_pick_chart["championName"].tolist()},
)
fig.show()

**Interpretation**

The most played Support champion in Bronze IV Solo 5x5 Ranked lobbies is Lux, followed by Morgana and Seraphine. This ranking remains the same as a previous iteration of this analysis, where only about 500 Match IDs were analyzed, but the gap between Lux and Morgana has widened significantly. Additionally, 4th and 5th place has changed, where they were previously Yuumi and Senna (respectively), now, it's Nami and Leona. 

From the data visualization above, we can also see a significant difference in players that chose conventional Support champions, and those that chose what we call an "off meta" pick, where a player selects a champion that wasn't designed to be a Support. I would say this involves every champion after and starting at Twitch, who is typically an ADC bottom lane pick. Players often do this to "troll", or purposefully play a bad game, or if they're really confident in their own skills. 

In future analysis, I will probably be filtering the data visualizations to only include champions that have more than 20 picks. 

In terms of why I selected this chart, I knew that there were going to be a high variety of differnet Support champions in this dataset, so I wanted to make sure that the relationship between all these different champions could be easily distinguishable. The big jump from Morgana to Lux is highly appreciated. The reader should be able to determine who the most picked Support champions are for Bronze IV ranked lobbies at a quick glance.

#### Question 2: What is the win rate for the top 3 Support champions in this lobby? 
The script below calculates the win rate for the top 3 most selected Support champions and sorts them in descending order.

In [30]:
champion_pick_counts = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean"))
    .sort_values("pick_count", ascending=False)
)

champion_pick_counts.head(3)

,championName,pick_count,win_rate
67,Lux,805,0.505590
77,Morgana,683,0.496340
108,Seraphine,668,0.483533


In [31]:
# Horizontal bar chart: win rates for the top 3 most-picked Supports (Question 2)
top_3_win_rates = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean"))
    .sort_values("pick_count", ascending=False)
    .head(3)
    .sort_values("win_rate", ascending=True)
)
top_3_win_rates["win_rate_pct"] = top_3_win_rates["win_rate"] * 100

fig = px.bar(
    top_3_win_rates,
    x="win_rate_pct",
    y="championName",
    orientation="h",
    title="Morgana has the highest win rate among the top 3 most-picked Bronze IV Supports",
    labels={"championName": "Support champion", "win_rate_pct": "Win rate (%)"},
    category_orders={"championName": top_3_win_rates["championName"].tolist()[::-1]},
)
fig.update_xaxes(range=[0, 100])
fig.show()

**Interpretation:**  
Lux has the highest win rate, but just barely. In a previous iteration of this analysis with a much smaller sample size, Morgana had the highest win rate by a significant amount. It's interesting to see these win rates stabilize by increasing the sample size. 

In terms of data visualization, I chose this chart type because there's only three values I wanted to represent and the horizontal bar graph allows users to scan its contents quicker, in a natural list order.

#### Question 3: Which of the top 10 Support champions with the most picks overall has the highest win rate?

The script below filters the dataset to include the top 10 champions with the most picks and sorts them in descending order based on win rate. 

In [49]:
champion_pick_counts = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean"))
    .query("pick_count > 20")
    .sort_values("win_rate", ascending=False)
)

champion_pick_counts

,championName,pick_count,win_rate
62,Leona,364,0.543956
117,Sona,311,0.517685
15,Brand,329,0.513678
81,Nautilus,343,0.513120
126,Thresh,302,0.509934
67,Lux,805,0.505590
77,Morgana,683,0.496340
92,Pyke,285,0.487719
108,Seraphine,668,0.483533
79,Nami,376,0.476064


In [50]:
# Scatter plot: Supports with 20+ picks, sorted by win rate (uses cell above)
if "champion_pick_counts" not in globals():
    raise RuntimeError("Run the cell above first so champion_pick_counts exists.")
if "win_rate" not in champion_pick_counts.columns:
    raise RuntimeError("Run a cell above that includes win_rate in champion_pick_counts.")

champion_pick_chart = champion_pick_counts.assign(
    win_rate_pct=lambda x: x["win_rate"] * 100
)

fig = px.scatter(
    champion_pick_chart,
    x="pick_count",
    y="win_rate_pct",
    text="championName",
    title="Win rate vs games played for Support champions with more than 20 picks",
    labels={
        "pick_count": "Games played",
        "win_rate_pct": "Win rate (%)",
        "championName": "Support champion",
    },
    hover_data={"pick_count": True, "win_rate_pct": ":.1f", "championName": True},
)
fig.update_traces(textposition="top center", marker=dict(size=12))
fig.update_yaxes(range=[0, 100])
fig.show()

**Interpretation**

With this scatter plot, we can see the huge jump in difference between the 3rd most played and 4th most played champions. However, the top 3 most played champions don't necessarily have the highest win rate. Within the top 5 champions with the highest win rate, 3 of them are Tank Supports (compared to top 3 most picked, which are all Mage Supports), with Leona in 1st place. This suggests that the game might be currently balanced to favor Tank Supports more, or that the nature of playing a Tank Support helps their ADC Bottom Lane partner do more damage. 

This actually reminds me of the time that my (much better) friend told me to go Tank because Tanks are better and I told him I didn't want to... I don't remember if we won. I guess this chart proves him right (infuriatingly so). 

In an initial iteration of this chart, I first created a bar chart similar to Question 1 because I thought it would best represent the difference in win rate between the top 10 champions. However, upon seeing this data actually visualized, I realized that the difference in win rate was far less dramatic than the difference in pick count. Additionally, the bar chart only represented the champion name on the x-axis and the win rate on the y-axis, ommitting an important piece of context: the difference in pick count between them. As a result, I selected a scatter plot in my second iteration, where all 3 pieces of this data could be represented at once. 

#### Question 3 Pt 2: Which Support champion with more than 20 picks has the highest win rate (not just restricted to the top 10 champions)? 

The script below filters the dataset to the champions with a pick count of more than 20 and sorts them in descending order based on win rate. 

In [11]:
champion_pick_counts = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean"))
    .query("pick_count > 20")
    .sort_values("win_rate", ascending=False)
)

champion_pick_counts.head(10)

,championName,pick_count,win_rate
135,Veigar,46,0.630435
5,Anivia,36,0.611111
119,Swain,271,0.605166
111,Shen,42,0.595238
154,Zoe,22,0.590909
39,Heimerdinger,22,0.590909
124,Taric,86,0.581395
70,Maokai,104,0.576923
110,Shaco,50,0.560000
44,Janna,58,0.551724


In [12]:
# Scatter plot: win rate vs. games played for Supports with 30+ picks (Question 3)
champion_win_rates = (
    df.groupby("championName", as_index=False)
    .agg(pick_count=("championName", "size"), win_rate=("win", "mean"))
    .query("pick_count >= 30")
    .assign(win_rate_pct=lambda d: d["win_rate"] * 100)
    .sort_values("win_rate", ascending=False)
)

fig = px.scatter(
    champion_win_rates,
    x="pick_count",
    y="win_rate_pct",
    text="championName",
    title="Win rate for champions with more than 30 picks",
    labels={
        "pick_count": "Games played",
        "win_rate_pct": "Win rate (%)",
        "championName": "Support champion",
    },
    hover_data={"pick_count": True, "win_rate_pct": ":.1f", "championName": True},
)
fig.update_traces(textposition="top center", marker=dict(size=12))
fig.update_yaxes(range=[0, 100])
fig.show()

**Interpretation:**  
When expanding the sample size to champions with a pick rate of more than 30, it's really interesting to see the volatility and high variation in win rate with champions that have less picks, albeit to be expected. I'm more interested in the selection of champions that have the highest win rate in this sample size, because 4 out of the top 5 picks (all except Swain) in this sorted dataset are technically "meta picks"; that is, champions that weren't technically designed to be Supports but have abilities that can still make them good picks, especially if someone has a lot of experience playing them. It's also interesting to note that Swain is also a Tank Support, and that none of the top 10 most picked champions appear in this list. 

I have a couple hypotheses for why this happens. The first one is because most of these champions are meta picks, they're seen a lot less often and Supports on the opposing team have a harder time understanding how to play around them. The second one is the presence of smurfs, or players that are actually ranked much higher on their main account that play in lower-level lobbies on a different account. A reason someone might do this could be because they want to play with friends that are in this rank, or it could be because they want to dunk on lower-level players and boost their own egos. Another reason why I've seen people do this is to rank up accounts for purchase, so that people online can buy these accounts once they've reached a certain rank for whatever reason. 

In terms of data visualization, my reasoning for selecting the scatter plot remains the same as Question 3 Pt 1, where the 3 most important pieces of data (champion name, win rate, and pick count) are contextualized all in one place. 

---

## Section 4 — Conclusions

This analysis definitely showed me what I was expecting, and even made me laugh because the top 3 support champions that I found from this analysis are also my personal top 3 champs to play (they're just really fun and have a lot of nice cosmetic skins). However, what surprised me was that the champions that are the most popular to play don't necessarily have the highest win rates. Based on this analysis, I would tell past me to start learning how to play Tank Supports if I really wanted to start ranking up, or at least learn Supports that would counter them really well. This suggests that my playing habits are typical for the average Bronze IV level player, but that Jeff (my higher ranked friend) was right when he said Tank Supports are way better picks (begrudgingly so...). 

There are a couple questions I would like to explore next: 

4. What is the distribution for the most played Support champion across all Level IV ranks (ex: Silver IV, Platinum IV, etc.)? 
5. What is the most common Bottom Lane/Support pairing in Bronze-level lobbies? What about across all ranks? 

The limitations of this analysis mostly have to do with the run times for the scripts to actually fetches the data from the API (due to Riot's API call limitations) and storage limitations due to the massive amounts of data that was getting returned. It took me around 3-4 hours to make all the API calls I needed to make statistically significant analysis, and even then my sample size is still pretty small. Additionally, all the data that's being used for this analysis is static, taken from a snapshot of my API calls from May 6th, 2026. If I were to repeat this process for other ranks, I would have to use snapshots from different dates and it would also take a really long time. 

--- 
## Section 5 - Process 

I've included a notebook that details my process in `MP1_dataset_prep.ipynb`, here's what it boils down to: 

1. Fetch PUUID based on Rank and Game Type (Bronze IV Solo 5x5 Ranked) from Summoner API
2. Fetch Match IDs based on PUUID from Match IDs API
3. Fetch Match Data based on Match IDs from Match Data API 
4. Filter fetched data down to include only Support champions and save to .csv file (`support_picks.csv`). 

Compared to my initial proposal, I had to adjust my analytical questions a few times because I didn't realize how long, how much work, and how much storage it would take to compile the final `support_picks.csv` file. However, I still believe I learned something valuable from my analysis despite this redirection. 